### Load Silver tables


In [0]:
# Load cleaned Silver tables

silver_orders = spark.table(
    "dbacademy.retailflow.silver_orders"
)

silver_order_items = spark.table(
    "dbacademy.retailflow.silver_order_items"
)

silver_products = spark.table(
    "dbacademy.retailflow.silver_products"
)

silver_stores = spark.table(
    "dbacademy.retailflow.silver_stores"
)

print("All Silver tables loaded successfully!")

All Silver tables loaded successfully!


In [0]:
print("Orders:", silver_orders.count())
print("Order Items:", silver_order_items.count())
print("Products:", silver_products.count())
print("Stores:", silver_stores.count())

Orders: 487
Order Items: 990
Products: 50
Stores: 10


In [0]:
from pyspark.sql.functions import col

# Join Orders with Order Items
sales_df = silver_orders.join(
    silver_order_items,
    on="order_id",
    how="inner"
)

# Join the result with Products
sales_df = sales_df.join(
    silver_products,
    on="product_id",
    how="inner"
)

# Join with Stores
sales_df = sales_df.join(
    silver_stores,
    on="store_id",
    how="inner"
)

print("Sales dataset created successfully!")

Sales dataset created successfully!


#### Calculate Revenue


In [0]:
from pyspark.sql.functions import round

gold_sales = sales_df.withColumn(
    "revenue",
    round(col("quantity") * col("price"), 2)
)

print("Revenue calculation completed!")

gold_sales.select(
    "order_id",
    "product_name",
    "category",
    "quantity",
    "price",
    "revenue",
    "city",
    "order_date"
).show(10, truncate=False)

Revenue calculation completed!
+--------+------------+---------+--------+-------+--------+----------+----------+
|order_id|product_name|category |quantity|price  |revenue |city      |order_date|
+--------+------------+---------+--------+-------+--------+----------+----------+
|11      |Product_21  |Groceries|4       |4164.08|16656.32|Hyderabad |2026-02-07|
|78      |Product_29  |Groceries|3       |897.01 |2691.03 |Bangalore |2026-02-28|
|470     |Product_23  |Groceries|2       |2929.03|5858.06 |Kolkata   |2026-02-17|
|159     |Product_36  |Unknown  |1       |1176.11|1176.11 |Coimbatore|2026-01-14|
|159     |Product_31  |Groceries|2       |3538.92|7077.84 |Coimbatore|2026-01-14|
|41      |Product_28  |Furniture|4       |3214.85|12859.4 |Mumbai    |2026-03-02|
|266     |Product_25  |Clothing |4       |3888.03|15552.12|Coimbatore|2026-01-15|
|103     |Product_8   |Unknown  |4       |1074.3 |4297.2  |Hyderabad |2026-03-16|
|178     |Product_34  |Clothing |2       |900.67 |1801.34 |Coimbato

In [0]:
from pyspark.sql.functions import (
    sum,
    countDistinct,
    avg,
    round,
    col
)

In [0]:
gold_daily_sales = gold_sales.groupBy(
    "order_date"
).agg(
    round(sum("revenue"), 2).alias("total_revenue"),

    countDistinct("order_id").alias("total_orders"),

    sum("quantity").alias("total_quantity_sold"),

    round(avg("revenue"), 2).alias("average_item_revenue")
)

gold_daily_sales = gold_daily_sales.orderBy("order_date")

print("Daily sales analytics created successfully!")

Daily sales analytics created successfully!


In [0]:
gold_daily_sales.show(20, truncate=False)

+----------+-------------+------------+-------------------+--------------------+
|order_date|total_revenue|total_orders|total_quantity_sold|average_item_revenue|
+----------+-------------+------------+-------------------+--------------------+
|2026-01-01|72875.14     |4           |31                 |9109.39             |
|2026-01-02|119012.76    |6           |41                 |9154.83             |
|2026-01-03|36945.95     |1           |15                 |12315.32            |
|2026-01-04|126397.51    |5           |46                 |9722.89             |
|2026-01-05|27713.4      |1           |9                  |13856.7             |
|2026-01-06|62335.62     |3           |25                 |7791.95             |
|2026-01-07|90017.04     |5           |36                 |8183.37             |
|2026-01-08|140282.95    |9           |59                 |6376.5              |
|2026-01-09|159270.38    |8           |75                 |6924.8              |
|2026-01-10|99611.89     |6 

In [0]:
gold_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbacademy.retailflow.gold_daily_sales")

print("Gold Daily Sales table created successfully!")

Gold Daily Sales table created successfully!


In [0]:
%sql

SELECT *
FROM dbacademy.retailflow.gold_daily_sales
ORDER BY order_date
LIMIT 20;

order_date,total_revenue,total_orders,total_quantity_sold,average_item_revenue
2026-01-01,72875.14,4,31,9109.39
2026-01-02,119012.76,6,41,9154.83
2026-01-03,36945.95,1,15,12315.32
2026-01-04,126397.51,5,46,9722.89
2026-01-05,27713.4,1,9,13856.7
2026-01-06,62335.62,3,25,7791.95
2026-01-07,90017.04,5,36,8183.37
2026-01-08,140282.95,9,59,6376.5
2026-01-09,159270.38,8,75,6924.8
2026-01-10,99611.89,6,41,7115.13


In [0]:
gold_product_performance = gold_sales.groupBy(
    "product_id",
    "product_name",
    "category"
).agg(
    sum("quantity").alias("total_quantity_sold"),

    round(sum("revenue"), 2).alias("total_revenue"),

    countDistinct("order_id").alias("total_orders"),

    round(avg("price"), 2).alias("average_product_price")
)

gold_product_performance = gold_product_performance.orderBy(
    col("total_revenue").desc()
)

print("Product performance analytics created successfully!")

Product performance analytics created successfully!


In [0]:
gold_product_performance.show(10, truncate=False)

+----------+------------+-----------+-------------------+-------------+------------+---------------------+
|product_id|product_name|category   |total_quantity_sold|total_revenue|total_orders|average_product_price|
+----------+------------+-----------+-------------------+-------------+------------+---------------------+
|37        |Product_37  |Groceries  |99                 |418767.03    |32          |4229.97              |
|25        |Product_25  |Clothing   |98                 |381026.94    |30          |3888.03              |
|41        |Product_41  |Clothing   |80                 |365949.6     |24          |4574.37              |
|50        |Product_50  |Clothing   |69                 |344096.1     |25          |4986.9               |
|42        |Product_42  |Unknown    |73                 |320776.6     |21          |4394.2               |
|28        |Product_28  |Furniture  |79                 |253973.15    |24          |3214.85              |
|21        |Product_21  |Groceries  |

In [0]:
gold_product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbacademy.retailflow.gold_product_performance")

print("Gold Product Performance table created successfully!")

Gold Product Performance table created successfully!


In [0]:
%sql

SELECT *
FROM dbacademy.retailflow.gold_product_performance
ORDER BY total_revenue DESC
LIMIT 10;

product_id,product_name,category,total_quantity_sold,total_revenue,total_orders,average_product_price
37,Product_37,Groceries,99,418767.03,32,4229.97
25,Product_25,Clothing,98,381026.94,30,3888.03
41,Product_41,Clothing,80,365949.6,24,4574.37
50,Product_50,Clothing,69,344096.1,25,4986.9
42,Product_42,Unknown,73,320776.6,21,4394.2
28,Product_28,Furniture,79,253973.15,24,3214.85
21,Product_21,Groceries,59,245680.72,18,4164.08
11,Product_11,Electronics,63,240544.08,20,3818.16
14,Product_14,Clothing,49,234726.66,17,4790.34
12,Product_12,Clothing,64,225336.32,18,3520.88


In [0]:
from pyspark.sql.functions import sum, countDistinct, round, col

gold_store_performance = gold_sales.groupBy(
    "store_id",
    "store_name",
    "city"
).agg(
    round(sum("revenue"), 2).alias("total_revenue"),

    countDistinct("order_id").alias("total_orders"),

    sum("quantity").alias("total_quantity_sold")
)

gold_store_performance = gold_store_performance.orderBy(
    col("total_revenue").desc()
)

print("Store performance analytics created successfully!")

Store performance analytics created successfully!


In [0]:
gold_store_performance.show(10, truncate=False)

+--------+----------+----------+-------------+------------+-------------------+
|store_id|store_name|city      |total_revenue|total_orders|total_quantity_sold|
+--------+----------+----------+-------------+------------+-------------------+
|8       |Store_8   |Kochi     |857815.9     |47          |336                |
|2       |Store_2   |Bangalore |848643.22    |49          |352                |
|9       |Store_9   |Kolkata   |838374.53    |48          |313                |
|4       |Store_4   |Mumbai    |756119.15    |49          |328                |
|7       |Store_7   |Coimbatore|737169.31    |47          |314                |
|5       |Store_5   |Delhi     |652794.6     |46          |289                |
|3       |Store_3   |Hyderabad |627577.33    |36          |261                |
|6       |Store_6   |Pune      |605206.2     |38          |240                |
|10      |Store_10  |Jaipur    |597709.96    |36          |243                |
|1       |Store_1   |Chennai   |551550.8

In [0]:
gold_store_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbacademy.retailflow.gold_store_performance")

print("Gold Store Performance table created successfully!")

Gold Store Performance table created successfully!


In [0]:
%sql

SELECT *
FROM dbacademy.retailflow.gold_store_performance
ORDER BY total_revenue DESC;

store_id,store_name,city,total_revenue,total_orders,total_quantity_sold
8,Store_8,Kochi,857815.9,47,336
2,Store_2,Bangalore,848643.22,49,352
9,Store_9,Kolkata,838374.53,48,313
4,Store_4,Mumbai,756119.15,49,328
7,Store_7,Coimbatore,737169.31,47,314
5,Store_5,Delhi,652794.6,46,289
3,Store_3,Hyderabad,627577.33,36,261
6,Store_6,Pune,605206.2,38,240
10,Store_10,Jaipur,597709.96,36,243
1,Store_1,Chennai,551550.83,33,213


In [0]:
from pyspark.sql.functions import (
    date_format,
    sum,
    countDistinct,
    avg,
    round,
    col
)

gold_monthly_sales = gold_sales.withColumn(
    "sales_month",
    date_format(col("order_date"), "yyyy-MM")
)

In [0]:
gold_monthly_sales = gold_monthly_sales.groupBy(
    "sales_month"
).agg(
    round(sum("revenue"), 2).alias("total_revenue"),

    countDistinct("order_id").alias("total_orders"),

    sum("quantity").alias("total_quantity_sold"),

    round(avg("revenue"), 2).alias("average_item_revenue")
)

gold_monthly_sales = gold_monthly_sales.orderBy("sales_month")

print("Monthly sales analytics created successfully!")

Monthly sales analytics created successfully!


In [0]:
gold_monthly_sales.show(truncate=False)

+-----------+-------------+------------+-------------------+--------------------+
|sales_month|total_revenue|total_orders|total_quantity_sold|average_item_revenue|
+-----------+-------------+------------+-------------------+--------------------+
|2026-01    |2553495.72   |146         |1021               |7488.26             |
|2026-02    |2212636.52   |135         |906                |7326.61             |
|2026-03    |2303010.63   |147         |961                |7130.06             |
|2026-04    |3818.16      |1           |1                  |3818.16             |
+-----------+-------------+------------+-------------------+--------------------+



In [0]:
gold_monthly_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbacademy.retailflow.gold_monthly_sales")

print("Gold Monthly Sales table created successfully!")

Gold Monthly Sales table created successfully!


In [0]:
%sql

SELECT *
FROM dbacademy.retailflow.gold_monthly_sales
ORDER BY sales_month;

sales_month,total_revenue,total_orders,total_quantity_sold,average_item_revenue
2026-01,2553495.72,146,1021,7488.26
2026-02,2212636.52,135,906,7326.61
2026-03,2303010.63,147,961,7130.06
2026-04,3818.16,1,1,3818.16


In [0]:
%sql

SHOW TABLES IN dbacademy.retailflow;

database,tableName,isTemporary
retailflow,bronze_order_items,false
retailflow,bronze_orders,false
retailflow,bronze_products,false
retailflow,bronze_stores,false
retailflow,gold_daily_sales,false
retailflow,gold_monthly_sales,false
retailflow,gold_product_performance,false
retailflow,gold_store_performance,false
retailflow,silver_order_items,false
retailflow,silver_orders,false


In [0]:
%sql

SELECT 'gold_daily_sales' AS table_name, COUNT(*) AS total_records
FROM dbacademy.retailflow.gold_daily_sales

UNION ALL

SELECT 'gold_product_performance', COUNT(*)
FROM dbacademy.retailflow.gold_product_performance

UNION ALL

SELECT 'gold_store_performance', COUNT(*)
FROM dbacademy.retailflow.gold_store_performance

UNION ALL

SELECT 'gold_monthly_sales', COUNT(*)
FROM dbacademy.retailflow.gold_monthly_sales;

table_name,total_records
gold_daily_sales,90
gold_product_performance,50
gold_store_performance,10
gold_monthly_sales,4
